# IT9203 — Azure Databricks Infrastructure Setup

**Project:** Top-Rated Movies per Genre Tracker  
**Dataset:** MovieLens 25M (GroupLens)  
**Platform:** Azure Databricks  
**Notebook:** 02 — Infrastructure Setup (10 marks)

## 1. Architecture Overview

The system runs on Azure Databricks with the following components:

**Pipeline Flow:**
DBFS (input CSVs) → Auto Loader (ingestion) → Apache Spark (processing + MLlib ALS) → Hive/Delta (SQL analytics) → Azure SQL Database (final output via JDBC)

**5 Big Data Tools:**
| # | Tool | Role |
|---|------|------|
| 1 | **DBFS + Databricks Cluster** | Distributed storage + resource mgmt (HDFS + YARN) |
| 2 | **Auto Loader** (Structured Streaming) | Incremental CSV ingestion |
| 3 | **Apache Spark** | Processing + MLlib ALS |
| 4 | **Apache Hive** (Spark SQL) | SQL analytics |
| 5 | **Azure SQL Database** | Final output store (JDBC) |

## 2. Prerequisites

Before running this notebook, ensure:

1. **Azure Databricks workspace** is created (portal.azure.com)
2. **Cluster is running** (Runtime 13.3 LTS, 2-4 nodes or single node)
3. **MovieLens CSVs are uploaded** to DBFS (`/FileStore/movielens/input/`)
   - `ratings.csv` (first 1M rows or full 25M)
   - `movies.csv`
4. **Azure SQL Database** is provisioned (movielens_analytics)
   - SQL user created (sqladmin)
   - Firewall rules configured for Databricks cluster IP
   - JDBC SQL Server driver installed on cluster

## 3. Verify Cluster and Environment

In [0]:
# Verify Spark version and cluster info
print(f"Spark version: {spark.version}")
print(f"Platform:      Azure Databricks (serverless)")
print(f"Cluster mode:  single node")

Spark version: 4.1.0
Platform:      Azure Databricks (serverless)
Cluster mode:  single node


## 4. Create DBFS Directory Structure

DBFS (Databricks File System) replaces HDFS as the distributed storage layer.

In [0]:
# Create project directory structure in DBFS
dbutils.fs.mkdirs("/FileStore/movielens/input/")
dbutils.fs.mkdirs("/FileStore/movielens/raw/")
dbutils.fs.mkdirs("/FileStore/movielens/processed/cleaned/")
dbutils.fs.mkdirs("/FileStore/movielens/processed/top_per_genre/")
dbutils.fs.mkdirs("/FileStore/movielens/output/als_recommendations/")
dbutils.fs.mkdirs("/FileStore/movielens/checkpoint/")

print("DBFS directories created successfully")
display(dbutils.fs.ls("/FileStore/movielens/"))

DBFS directories created successfully


path,name,size,modificationTime
dbfs:/FileStore/movielens/checkpoint/,checkpoint/,0,1779546492000
dbfs:/FileStore/movielens/input/,input/,0,1779546159000
dbfs:/FileStore/movielens/output/,output/,0,1779546492000
dbfs:/FileStore/movielens/processed/,processed/,0,1779546492000
dbfs:/FileStore/movielens/raw/,raw/,0,1779546491000


## 5. Upload Data to DBFS

Run this cell to verify data files exist in DBFS. If empty, upload via Databricks UI:
- Data tab -> Add Data -> Upload `ratings.csv` and `movies.csv`
- Then copy to the input directory

In [0]:
# Check if data files exist
files = dbutils.fs.ls("/FileStore/movielens/input/")
if len(files) == 0:
    print("No files found. Upload ratings.csv and movies.csv via Databricks UI:")
    print("1. Data tab -> Add Data -> Upload files")
    print("2. Select both CSV files")
    print("3. After upload, use dbutils.fs.cp() to move them to /FileStore/movielens/input/")
    print("\nOR run the next cell if files are already uploaded elsewhere")
else:
    print("Files found:")
    for f in files:
        print(f"  {f.name} ({f.size / 1e6:.1f} MB)")

Files found:
  movies.csv (3.0 MB)
  ratings.csv (678.3 MB)


In [0]:
# If you uploaded to a different DBFS path, copy files here
# dbutils.fs.cp("dbfs:/FileStore/ratings.csv", "/FileStore/movielens/input/ratings.csv")
# dbutils.fs.cp("dbfs:/FileStore/movies.csv", "/FileStore/movielens/input/movies.csv")
print("Uncomment and modify paths above if needed")

Uncomment and modify paths above if needed


## 6. Create Hive Database

Hive metadata is managed by Databricks via the Hive metastore. We create the `movielens` database here.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS movielens")
spark.sql("USE movielens")
spark.sql("SHOW DATABASES").show()

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
|         movielens|
+------------------+



## 7. Auto Loader Configuration

Auto Loader is Databricks' incremental file ingestion tool. It monitors a directory for new CSV files and processes them as a stream. We configure it here and will run it in Notebook 03.

**Schema:**
| Column | Type | Source |
|---|---|---|
| userId | int | ratings.csv |
| movieId | int | Both files |
| rating | double | ratings.csv |
| timestamp | long | ratings.csv |
| title | string | movies.csv |
| genres | string | movies.csv |

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType, StringType

# Define the schema for Auto Loader
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True)
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

print("Schemas defined.")
print("Auto Loader will ingest new CSV files from /FileStore/movielens/input/")
print("and land them as Parquet in /FileStore/movielens/raw/")

Schemas defined.
Auto Loader will ingest new CSV files from /FileStore/movielens/input/
and land them as Parquet in /FileStore/movielens/raw/


## 8. Azure SQL Database Setup (Final Output)

Azure SQL Database is used as the final output store. Set up via Azure Portal:

1. Go to https://portal.azure.com
2. Create a new SQL Database (movielens_analytics)
3. Create a SQL Server with admin credentials (sqladmin)
4. Under Networking, enable public network access
5. Add a firewall rule for the Databricks cluster IP
6. Install JDBC driver on Databricks cluster (com.microsoft.sqlserver:mssql-jdbc:12.2.0.jre11)

**JDBC connection string format:**

jdbc:sqlserver://<server>.database.windows.net:1433;database=movielens_analytics;encrypt=true;

This connection will be used in Notebook 06.

In [0]:
SERVER = "movielenssql202508993.database.windows.net"
DATABASE = "movielens_analytics"
USERNAME = "sqladmin"
PORT = 1433

JDBC_URL = f"jdbc:sqlserver://{SERVER}:{PORT};database={DATABASE};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

print(f"Azure SQL target: {DATABASE}")
print(f"Server: {SERVER}")
print(f"JDBC URL configured: True")

Azure SQL target: movielens_analytics
Server: movielenssql202508993.database.windows.net
JDBC URL configured: True


## 9. Infrastructure Summary

All infrastructure is now configured:

| Component | Status | Details |
|---|---|---|
| Azure Databricks Cluster | Ready | Spark 4.1.0 on single node |
| DBFS storage | Ready | 6 directories created under /FileStore/movielens/ |
| Dataset | Ready | ratings.csv + movies.csv in /FileStore/movielens/input/ |
| Hive Metastore | Ready | Database movielens created |
| Auto Loader | Configured | Schema defined, checkpoint dir set |
| Azure SQL Database | Configured | movielenssql202508993.database.windows.net |

**Proceed to Notebook 03 — Data Ingestion.**